# Algorithmic Trading, Backtesting, and Performance Evaluation

## Learning objectives

By the end of this notebook, you should be able to:

- Translate an indicator into clear trading rules.
- Implement a transparent manual backtest.
- Compute core performance and risk metrics.
- Compare a manual workflow with a specialized framework (`vectorbt`).

## Where this notebook fits in the course

This is the final notebook in the sequence. After data analysis (Notebook 1) and model architecture (Notebook 2), we now build an end-to-end quantitative trading workflow.

## Notebook narrative

1. Acquire and prepare market data.
2. Construct an indicator and define trading signals.
3. Run a manual backtest with explicit assumptions.
4. Evaluate performance and drawdowns.
5. Re-run the strategy with `vectorbt` and compare outputs.


## Libraries used and why

- `pandas` and `numpy`: time-series calculations and metrics.
- `matplotlib` and `seaborn`: clean performance visualization.
- `yfinance`: historical market data retrieval.
- `vectorbt`: specialized, standardized backtesting workflow.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import yfinance as yf
import vectorbt as vbt
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook", palette="deep")
plt.rcParams["figure.figsize"] = (11, 5)
plt.rcParams["axes.titleweight"] = "bold"
pd.set_option("display.float_format", "{:.4f}".format)


## Trading workflow overview

**data -> indicator -> signal -> position -> returns -> performance**

This explicit pipeline helps avoid hidden assumptions and makes model credibility easier to assess.


## 1. Data acquisition

We use `SPY` as a liquid benchmark series with long historical coverage.


In [ ]:
ticker = "SPY"
start_date = "2010-01-01"
end_date = pd.Timestamp.today().strftime("%Y-%m-%d")

raw = yf.download(ticker, start=start_date, end=end_date, auto_adjust=False, progress=False)

if raw.empty:
    raise ValueError("No data returned by yfinance. Verify internet access and ticker symbol.")

close = raw["Adj Close"] if "Adj Close" in raw.columns else raw["Close"]
if isinstance(close, pd.DataFrame):
    # yfinance can return a single-column DataFrame; convert to Series for cleaner downstream stats.
    close = close.iloc[:, 0]
close = close.rename(ticker)

close.tail()


### Section transition

With data in place, we can define a simple indicator-based strategy and focus on implementation quality.


## 2. Indicator construction (SMA crossover)

- Short moving average: 50 trading days
- Long moving average: 200 trading days


In [ ]:
short_window = 50
long_window = 200

signals = pd.DataFrame(index=close.index)
signals["close"] = close
signals["sma_short"] = close.rolling(short_window).mean()
signals["sma_long"] = close.rolling(long_window).mean()

signals.tail()


## 3. Signal generation and position logic

We use a long/flat rule:

- `1` when `sma_short > sma_long`
- `0` otherwise

To prevent look-ahead bias, the trading position is shifted by one day.


In [ ]:
signals["signal"] = (signals["sma_short"] > signals["sma_long"]).astype(int)
signals["position"] = signals["signal"].shift(1).fillna(0)

signals[["sma_short", "sma_long", "signal", "position"]].dropna().tail()


### Discussion

Timing conventions are not cosmetic details. A backtest without explicit signal-to-trade timing can overstate performance.

### Section transition

Now we implement the full manual return pipeline before introducing any specialized framework.


## 4. Manual backtest logic


In [ ]:
backtest = pd.DataFrame(index=signals.index)
backtest["close"] = signals["close"]
backtest["position"] = signals["position"]
backtest["market_return"] = backtest["close"].pct_change().fillna(0)
backtest["strategy_return"] = backtest["position"] * backtest["market_return"]

backtest["equity_strategy"] = (1 + backtest["strategy_return"]).cumprod()
backtest["equity_buy_hold"] = (1 + backtest["market_return"]).cumprod()

backtest.tail()


## 5. Performance metrics (manual)

Metrics reported:

- Annualized return
- Annualized volatility
- Sharpe ratio (risk-free rate set to 0 for simplicity)
- Maximum drawdown


In [ ]:
def compute_metrics(returns: pd.Series, equity_curve: pd.Series) -> dict:
    periods = 252
    ann_return = (1 + returns).prod() ** (periods / len(returns)) - 1
    ann_vol = returns.std() * np.sqrt(periods)
    sharpe = ann_return / ann_vol if ann_vol > 0 else np.nan

    drawdown = equity_curve / equity_curve.cummax() - 1
    max_drawdown = drawdown.min()

    return {
        "Annualized Return": ann_return,
        "Annualized Volatility": ann_vol,
        "Sharpe Ratio": sharpe,
        "Max Drawdown": max_drawdown,
    }

manual_strategy_metrics = compute_metrics(backtest["strategy_return"], backtest["equity_strategy"])
buy_hold_metrics = compute_metrics(backtest["market_return"], backtest["equity_buy_hold"])

manual_metrics = pd.DataFrame([manual_strategy_metrics, buy_hold_metrics], index=["SMA Strategy", "Buy & Hold"])
manual_metrics


## 6. Visualization of manual results


In [ ]:
fig, ax = plt.subplots()
ax.plot(signals.index, signals["close"], label="Price", linewidth=1.7, color="#1f77b4")
ax.plot(signals.index, signals["sma_short"], label=f"SMA {short_window}", linewidth=1.3, color="#ff7f0e")
ax.plot(signals.index, signals["sma_long"], label=f"SMA {long_window}", linewidth=1.3, color="#2ca02c")
ax.set_title(f"{ticker}: Price and SMA Crossover Inputs")
ax.set_xlabel("Date")
ax.set_ylabel("Price (USD)")
ax.legend(frameon=True)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots()
ax.plot(backtest.index, backtest["equity_strategy"], label="SMA Strategy", linewidth=1.7, color="#1f77b4")
ax.plot(backtest.index, backtest["equity_buy_hold"], label="Buy & Hold", linewidth=1.7, color="#7f7f7f")
ax.set_title("Equity Curves: Strategy vs Buy-and-Hold (Base = 1)")
ax.set_xlabel("Date")
ax.set_ylabel("Portfolio Value")
ax.legend(frameon=True)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
strategy_dd = backtest["equity_strategy"] / backtest["equity_strategy"].cummax() - 1
buy_hold_dd = backtest["equity_buy_hold"] / backtest["equity_buy_hold"].cummax() - 1

fig, ax = plt.subplots()
ax.plot(strategy_dd.index, strategy_dd, label="SMA Strategy", linewidth=1.5, color="#1f77b4")
ax.plot(buy_hold_dd.index, buy_hold_dd, label="Buy & Hold", linewidth=1.5, color="#7f7f7f")
ax.set_title("Drawdown Profile Comparison")
ax.set_xlabel("Date")
ax.set_ylabel("Drawdown")
ax.legend(frameon=True)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


### Discussion

Manual implementation is pedagogically valuable because every assumption is visible. It also builds intuition for what specialized backtesting libraries are automating.

### Section transition

We now run the same strategy idea with `vectorbt` to compare workflows.


## 7. Introduce vectorbt

`vectorbt` allows faster strategy iteration with standardized execution and reporting tools.


In [ ]:
entries = signals["sma_short"].vbt.crossed_above(signals["sma_long"])
exits = signals["sma_short"].vbt.crossed_below(signals["sma_long"])

portfolio = vbt.Portfolio.from_signals(
    close,
    entries=entries,
    exits=exits,
    init_cash=100_000,
    fees=0.0005,
    slippage=0.0005,
    freq="1D",
)

vbt_stats = portfolio.stats()
vbt_stats


## 8. Compare manual workflow vs vectorbt workflow

Manual workflow emphasizes transparency. `vectorbt` emphasizes consistency and scalability.


In [ ]:
vbt_returns = portfolio.returns()
if isinstance(vbt_returns, pd.DataFrame):
    vbt_returns = vbt_returns.iloc[:, 0]

vbt_equity = portfolio.value()
if isinstance(vbt_equity, pd.DataFrame):
    vbt_equity = vbt_equity.iloc[:, 0]

periods = 252
vbt_ann_return = (1 + vbt_returns).prod() ** (periods / len(vbt_returns)) - 1
vbt_ann_vol = vbt_returns.std() * np.sqrt(periods)
vbt_sharpe = vbt_stats.get("Sharpe Ratio", np.nan)
if pd.isna(vbt_sharpe):
    vbt_sharpe = vbt_ann_return / vbt_ann_vol if vbt_ann_vol > 0 else np.nan

# Keep drawdown sign convention consistent with manual section (negative drawdowns).
vbt_drawdown = vbt_equity / vbt_equity.cummax() - 1
vbt_max_drawdown = vbt_drawdown.min()

comparison = pd.DataFrame({
    "Manual (SMA)": manual_metrics.loc["SMA Strategy"],
    "vectorbt": [
        vbt_ann_return,
        vbt_ann_vol,
        vbt_sharpe,
        vbt_max_drawdown,
    ],
}, index=["Annualized Return", "Annualized Volatility", "Sharpe Ratio", "Max Drawdown"])

comparison


## 9. Interpretation

- Backtest credibility depends on clear timing rules and realistic transaction assumptions.
- Simplicity is a feature in teaching: clear logic is easier to validate and discuss.
- Overfitting remains a major risk when parameters are tuned without strong economic rationale.

### Discussion

A robust quantitative workflow combines manual transparency (for understanding) with specialized tooling (for scale and consistency).


## Key takeaways

- Quant strategy research should follow a visible and reproducible pipeline.
- Performance metrics are only meaningful when implementation assumptions are explicit.
- Specialized frameworks add speed and standardization, but do not replace critical financial interpretation.

## End-of-repository concluding note

Across the three notebooks, the central skill is workflow discipline: define the financial question, structure the pipeline, and interpret outputs with economic judgment. This is the core of practical Financial Engineering in Python.
